### Required Task 16 Summary

This document details the development and testing of an AI agent using the Gemini 2.5 Flash model, enhanced with custom financial analysis tools.

**1. Steps Taken:**
*   **Setup:** Configured the Gemini API and installed `yfinance`.
*   **Tool Definition:** Created three Python functions: `get_stock_price`, `get_company_risk_score`, and `get_pe_ratio`.
*   **Model Initialization:** Initialized `gemini-2.5-flash` with these tools, enabling automatic function calling.
*   **Agent Testing:** Tested with single-tool (stock price), multi-tool (price and risk), and complex multi-tool queries (Task 16: P/E ratio, price, and risk for valuation comparison).
*   **Inspection:** Reviewed chat history to understand the agent's decision-making and tool usage.

**2. Results & Evaluation:**
*   **Successful Tool Integration & Reasoning:** The agent seamlessly integrated and effectively reasoned with all tools, accurately executing them and synthesizing information for single and multi-tool queries.
*   **Comprehensive Analysis:** For complex queries (like Task 16's valuation), the agent successfully combined data from multiple tools to provide insightful and accurate financial assessments.
*   **Observability:** The chat history confirmed the agent's dynamic function calling and integration of tool outputs into its responses.

Overall, the experiment demonstrated the strong capabilities of Generative AI agents in complex, tool-augmented tasks.

Agentic AI is the step where the model transitions from a "passive knowledge engine" to an "active operator" that can use tools, perform actions, and make multi-step decisions.

#Concepts & The "Tool-Using" Brain


**Agents vs. RAG:** RAG is about context, while Agents are about action.

**The ReAct Pattern:** Reasoning + Acting. The loop where the AI thinks, selects a tool, executes it, observes the output, and thinks again.

**Tools:** What is a "tool"? (A Python function that the AI can "call").

### Setup and API Configuration
This cell handles the initial setup. It installs the necessary `google-generativeai` library, imports it, and then configures the Gemini API using your API key. The API key is securely retrieved from Colab's user data secrets.

In [3]:
import google.generativeai as genai
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

### Install `yfinance` Library
This cell installs the `yfinance` library, which is used to fetch financial data like stock prices and company information from Yahoo Finance. The `-q` flag ensures a quiet installation, and `-U` updates the package if it's already installed.

In [4]:
!pip install -q yfinance

### Define Custom Tools (Python Functions)
Here, we define two Python functions that will act as 'tools' for our Gemini agent:

1.  **`get_stock_price(ticker)`**: Takes a stock ticker symbol (e.g., 'AAPL') and returns its current live price.
2.  **`get_company_risk_score(ticker)`**: Calculates a simplified risk assessment for a given stock based on its Beta value, which measures market volatility. It categorizes risk as 'High', 'Moderate', or 'Low'.

In [5]:
import yfinance as yf

# Define the "Real" Tools

def get_stock_price(ticker: str):
    """
    Retrieves the current live stock price for a given ticker.
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL', 'NVDA').
    """
    print(f"  ... 🔍 TOOL CALL: Connecting to Yahoo Finance for {ticker} ...")

    try:
        stock = yf.Ticker(ticker)
        # .fast_info is much faster than .info for just price
        price = stock.fast_info['last_price']
        return round(price, 2)
    except Exception as e:
        return f"Error fetching price for {ticker}: {e}"

def get_company_risk_score(ticker: str):
    """
    Calculates a risk proxy based on the stock's Beta (market volatility).
    A Beta > 1.0 means higher risk/volatility than the market.
    Args:
        ticker: The stock ticker symbol.
    """
    print(f"  ... ⚠️ TOOL CALL: Fetching Risk Metrics for {ticker} ...")

    try:
        stock = yf.Ticker(ticker)
        beta = stock.info.get('beta', 0)

        # We can add some logic to make it more "human-readable" for the agent
        if beta > 1.5:
            assessment = "High Risk (High Volatility)"
        elif beta < 0.8:
            assessment = "Low Risk (Stable)"
        else:
            assessment = "Moderate Risk"

        return {"beta": beta, "assessment": assessment}
    except Exception as e:
        return "Risk data unavailable"

print("✅ Real-Time Tools Defined.")

✅ Real-Time Tools Defined.


### Initialize the Generative Model with Tools
This cell is crucial for enabling the agentic capabilities. This is where the model gets its "hands." We pass the functions directly to the tools parameter. We:

1.  Specify `gemini-2.5-flash` as our model, known for its speed and tool-use abilities.
2.  Pass our custom `tools_list` (containing `get_stock_price` and `get_company_risk_score`) to the model during initialization.
3.  Start a chat session (`model.start_chat`) and enable `enable_automatic_function_calling=True`. This tells the SDK to automatically execute any tool calls suggested by the model during the conversation and feed the results back to the model, streamlining the agent's thought process.

In [6]:
# 4. Initialize Model with Tools
# We use 'gemini-2.5-flash' as it is fast and excellent at tool use.

tools_list = [get_stock_price, get_company_risk_score]

model = genai.GenerativeModel(
    "gemini-2.5-flash",
    tools=tools_list
)

# Enable automatic function calling
# This tells the SDK: "If the model asks to run a function, just run it for me and give the answer back."
chat = model.start_chat(enable_automatic_function_calling=True)

print("✅ Model initialized with Agentic capabilities.")

✅ Model initialized with Agentic capabilities.


### Test: Single Tool Call
This cell demonstrates a simple interaction where the model needs to use only one tool (`get_stock_price`) to answer the user's query. The agent will identify the need for this tool, call it with the correct argument ('AAPL'), and then formulate a response based on the tool's output.

In [7]:
# 5. Test: Single Tool Call
response = chat.send_message("What is the current stock price of Apple?")

print("\n🤖 AGENT RESPONSE:")
print(response.text)

  ... 🔍 TOOL CALL: Connecting to Yahoo Finance for AAPL ...

🤖 AGENT RESPONSE:
The current stock price of Apple is $247.99.


### Test: Multi-Step Reasoning with Multiple Tools
This is a more complex example. The user's query requires the model to combine information from *two* different tools (`get_stock_price` and `get_company_risk_score`) to provide a comprehensive answer and investment advice. The agent will determine that both pieces of information are needed, execute both tools, and then synthesize the results into a coherent response.

In [8]:
# 6. Test: Multi-Step Reasoning
# The model should call get_stock_price AND get_company_risk_score

query = "I am conservative investor. Should I buy Apple stock? Tell me its price and risk score."
print(f"❓ USER QUERY: {query}\n")

response = chat.send_message(query)

print("\n🤖 AGENT RESPONSE:")
print(response.text)

❓ USER QUERY: I am conservative investor. Should I buy Apple stock? Tell me its price and risk score.

  ... 🔍 TOOL CALL: Connecting to Yahoo Finance for AAPL ...
  ... ⚠️ TOOL CALL: Fetching Risk Metrics for AAPL ...

🤖 AGENT RESPONSE:
Apple's current stock price is $247.99. Its risk score is moderate, with a beta of 1.116. A beta greater than 1.0 indicates higher volatility compared to the overall market. As a conservative investor, you should consider that Apple's stock has a moderate risk profile due to its higher volatility.


### Inspecting the Agent's Thought Process
This cell allows us to peek 'under the hood' of the agent's actions. By iterating through the `chat.history`, we can see the sequence of messages exchanged, including:

*   **User queries**
*   **Model's function calls (🔧 ACTION)**: When the model decides to use a tool, this shows which tool it called and with what arguments.
*   **Tool's observations (📨 OBSERVATION)**: The output returned by the executed tool, which the model then uses for its next step of reasoning.
*   **Model's final responses**: The natural language answers provided by the model.

In [9]:
# 7. Inspection: What actually happened?
# We can look at the chat history to see the "FunctionCall" objects.

print("🕵️‍♂️ HISTORY INSPECTION:\n")
for message in chat.history:
    role = message.role
    print(f"--- {role.upper()} ---")

    # Check if there are function calls
    for part in message.parts:
        if fn := part.function_call:
            print(f"🔧 ACTION: Calling Function '{fn.name}' with args: {dict(fn.args)}")
        elif resp := part.function_response:
            print(f"📨 OBSERVATION: Function returned: {resp.response}")
        else:
            print(part.text)

🕵️‍♂️ HISTORY INSPECTION:

--- USER ---
What is the current stock price of Apple?
--- MODEL ---
🔧 ACTION: Calling Function 'get_stock_price' with args: {'ticker': 'AAPL'}
--- USER ---
📨 OBSERVATION: Function returned: <proto.marshal.collections.maps.MapComposite object at 0x7b03716a9cd0>
--- MODEL ---
The current stock price of Apple is $247.99.
--- USER ---
I am conservative investor. Should I buy Apple stock? Tell me its price and risk score.
--- MODEL ---
🔧 ACTION: Calling Function 'get_stock_price' with args: {'ticker': 'AAPL'}
🔧 ACTION: Calling Function 'get_company_risk_score' with args: {'ticker': 'AAPL'}
--- USER ---
📨 OBSERVATION: Function returned: <proto.marshal.collections.maps.MapComposite object at 0x7b03716a9b80>
📨 OBSERVATION: Function returned: <proto.marshal.collections.maps.MapComposite object at 0x7b03716a88f0>
--- MODEL ---
Apple's current stock price is $247.99. Its risk score is moderate, with a beta of 1.116. A beta greater than 1.0 indicates higher volatility c

#Try The "Comparative" Challenge

Without changing any code, prompt the agent to compare Microsoft (MSFT) and NVIDIA (NVDA). Ask it specifically which one is 'riskier' based on the tools you have.




# Required Task 16

Add a new tool called `get_pe_ratio` that fetches the Price-to-Earnings ratio. Then ask the agent if Apple is 'overvalued' compared to the average market P/E of 25.

**Hint:**
```
stock = yf.Ticker(ticker)
# PE ratio is often in 'summaryDetail' or 'info'
 pe = stock.info.get('trailingPE', 'N/A')
```




### Solution: Adding the `get_pe_ratio` Tool
We define a new tool function that fetches the trailing P/E ratio from Yahoo Finance. Then we reinitialize the model with all three tools and ask the agent to evaluate whether Apple is overvalued.

In [10]:
# --- NEW TOOL: Price-to-Earnings Ratio ---

def get_pe_ratio(ticker: str):
    """
    Fetches the trailing Price-to-Earnings (P/E) ratio for a given stock ticker.
    The P/E ratio indicates how much investors pay per dollar of earnings.
    A higher P/E may suggest the stock is overvalued relative to its earnings.
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL', 'MSFT').
    """
    print(f"  ... 📊 TOOL CALL: Fetching P/E Ratio for {ticker} ...")

    try:
        stock = yf.Ticker(ticker)
        pe = stock.info.get('trailingPE', 'N/A')
        if pe != 'N/A':
            pe = round(pe, 2)
        return {"ticker": ticker, "trailing_PE": pe}
    except Exception as e:
        return f"Error fetching P/E ratio for {ticker}: {e}"

print("✅ New Tool 'get_pe_ratio' Defined.")

✅ New Tool 'get_pe_ratio' Defined.


In [11]:
# Reinitialize the model with all THREE tools

tools_list = [get_stock_price, get_company_risk_score, get_pe_ratio]

model = genai.GenerativeModel(
    "gemini-2.5-flash",
    tools=tools_list
)

chat = model.start_chat(enable_automatic_function_calling=True)

print("✅ Model reinitialized with 3 tools: get_stock_price, get_company_risk_score, get_pe_ratio")

✅ Model reinitialized with 3 tools: get_stock_price, get_company_risk_score, get_pe_ratio


In [12]:
# Ask the agent to evaluate Apple's valuation using the new P/E tool

query = (
    "Is Apple (AAPL) overvalued? Get its P/E ratio and compare it to the "
    "average market P/E of 25. Also tell me the current stock price and risk score."
)
print(f"❓ USER QUERY: {query}\n")

response = chat.send_message(query)

print("\n🤖 AGENT RESPONSE:")
print(response.text)

❓ USER QUERY: Is Apple (AAPL) overvalued? Get its P/E ratio and compare it to the average market P/E of 25. Also tell me the current stock price and risk score.

  ... 📊 TOOL CALL: Fetching P/E Ratio for AAPL ...
  ... 🔍 TOOL CALL: Connecting to Yahoo Finance for AAPL ...
  ... ⚠️ TOOL CALL: Fetching Risk Metrics for AAPL ...

🤖 AGENT RESPONSE:
Apple's P/E ratio is 31.35, which is higher than the average market P/E of 25. This might suggest that Apple is currently overvalued.

Its current stock price is $247.99 and its risk score is "Moderate Risk" with a beta of 1.116. A beta greater than 1 indicates that Apple's stock is more volatile than the overall market.


In [13]:
# Inspect the agent's thought process for the valuation query

print("🕵️‍♂️ HISTORY INSPECTION:\n")
for message in chat.history:
    role = message.role
    print(f"--- {role.upper()} ---")

    for part in message.parts:
        if fn := part.function_call:
            print(f"🔧 ACTION: Calling Function '{fn.name}' with args: {dict(fn.args)}")
        elif resp := part.function_response:
            print(f"📨 OBSERVATION: Function returned: {resp.response}")
        else:
            print(part.text)

🕵️‍♂️ HISTORY INSPECTION:

--- USER ---
Is Apple (AAPL) overvalued? Get its P/E ratio and compare it to the average market P/E of 25. Also tell me the current stock price and risk score.
--- MODEL ---
🔧 ACTION: Calling Function 'get_pe_ratio' with args: {'ticker': 'AAPL'}
🔧 ACTION: Calling Function 'get_stock_price' with args: {'ticker': 'AAPL'}
🔧 ACTION: Calling Function 'get_company_risk_score' with args: {'ticker': 'AAPL'}
--- USER ---
📨 OBSERVATION: Function returned: <proto.marshal.collections.maps.MapComposite object at 0x7b0383f78ef0>
📨 OBSERVATION: Function returned: <proto.marshal.collections.maps.MapComposite object at 0x7b0383f78ef0>
📨 OBSERVATION: Function returned: <proto.marshal.collections.maps.MapComposite object at 0x7b0383f78ef0>
--- MODEL ---
Apple's P/E ratio is 31.35, which is higher than the average market P/E of 25. This might suggest that Apple is currently overvalued.

Its current stock price is $247.99 and its risk score is "Moderate Risk" with a beta of 1.116